# 03 · Results

This notebook displays and evaluates the probabilistic forecast produced in [notebook 02](02_training_and_predicting.ipynb):

1. the **hydrograph** with predictive uncertainty bands, against observations and HYPE's own best calibration;
2. the **reliability** of the ensemble (PIT / Q-Q diagram, interval coverage and sharpness);
3. **skill metrics** (NSE, KGE, percent bias) for the ensemble median vs the deterministic HYPE baseline.

It reads `examples/predictions.csv` written by notebook 02 — run that notebook first.

In [ ]:
# --- Bootstrap: run from the repo root ---
import os, sys
from pathlib import Path

def find_repo_root(start=None):
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "gpu.py").exists():
            return p
    raise RuntimeError("Could not find the repo root above %s" % start)

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
%matplotlib inline

from gpu import GPU
print("Imports OK")

## 1 · Load the forecast, observations and baselines

In [ ]:
model = GPU.load("examples/GPU_HYPE_2026-03-24_13h15.pkl")
bands = np.array(model.opt["bands"])

pred_path = Path("examples/predictions.csv")
if not pred_path.exists():
    raise FileNotFoundError("Run notebook 02 first to create examples/predictions.csv")
predictions = pd.read_csv(pred_path, header=[0, 1], index_col=0, parse_dates=True)
records = predictions.index
band_arr = predictions.values            # (n_steps, n_bands)
print("forecast window:", records[0].date(), "\u2192", records[-1].date(), "| bands:", band_arr.shape[1])

# Observations on the same window
obs = pd.read_csv("examples/Qobs.txt", sep="\t", header=0,
                  index_col="Date", parse_dates=True)["50675"].reindex(records)

# HYPE's own best (deterministic) calibration, as a baseline
best_hype = pd.read_csv("examples/BestAutomaticCalibration_0050675.txt", sep="\t",
                        skiprows=[1], header=0, index_col=0, parse_dates=True)
best_hype = best_hype.iloc[:, 0].reindex(records)

In [ ]:
def band_col(level):
    """Column of the forecast nearest a given non-exceedance probability."""
    return band_arr[:, int(np.argmin(np.abs(bands - level)))]

median = 0.5 * (band_col(0.45) + band_col(0.55))

## 2 · Hydrograph with uncertainty bands

The shaded envelopes are the ensemble's predictive intervals. A skilful probabilistic forecast keeps the observations inside the bands most of the time while staying sharp (narrow).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(records, band_col(0.025), band_col(0.975), color="tab:blue", alpha=0.15, label="2.5\u201397.5%")
ax.fill_between(records, band_col(0.05), band_col(0.95), color="tab:blue", alpha=0.25, label="5\u201395%")
ax.fill_between(records, band_col(0.25), band_col(0.75), color="tab:blue", alpha=0.40, label="25\u201375%")
ax.plot(records, median, color="tab:blue", lw=1.0, label="GPU_HYPE median")
ax.plot(records, best_hype.values, color="tab:green", lw=0.8, label="Best HYPE (deterministic)")
ax.plot(records, obs.values, color="black", lw=0.8, label="Observed")
ax.set_ylabel("Discharge [m\u00b3/s]"); ax.set_title("Validation forecast \u2014 T\u00fcrkheim (subbasin 50675)")
ax.legend(ncol=3, fontsize=9)
plt.tight_layout()

## 3 · Reliability (PIT / Q-Q), coverage and sharpness

For each day we locate the observation within the forecast's quantile bands — its **probability-integral-transform (PIT)** value. A well-calibrated ensemble produces PIT values that are uniform on [0, 1], so the *sorted* PIT curve follows the diagonal.

We also report two summary numbers: the empirical **coverage** of the 5–95% interval (ideally ≈ 90%) and the mean **width** of the 25–75% band (sharpness — narrower is better, given good coverage).

In [ ]:
def pit_value(o, band_values, levels):
    """Where observation `o` falls in the forecast CDF (clipped to [0, 1])."""
    if not np.isfinite(o):
        return np.nan
    if o <= band_values[0]:
        return 0.0
    if o >= band_values[-1]:
        return 1.0
    return float(np.interp(o, band_values, levels))

pit = np.array([pit_value(o, band_arr[i], bands) for i, o in enumerate(obs.values)])
pit = np.sort(pit[np.isfinite(pit)])
uniform = np.linspace(0, 1, len(pit))

# Coverage of the 5-95% interval and mean 25-75% width (sharpness)
lo, hi = band_col(0.05), band_col(0.95)
finite = np.isfinite(obs.values)
coverage = 100 * np.mean(((obs.values >= lo) & (obs.values <= hi))[finite])
sharpness = float(np.nanmean(band_col(0.75) - band_col(0.25)))
print(f"5-95% coverage: {coverage:.1f}%  (target ~90%)   |   mean 25-75% width: {sharpness:.2f} m³/s")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
ax.plot(uniform, pit, color="tab:blue", lw=2, label="GPU_HYPE")
ax.set_xlabel("Theoretical quantile"); ax.set_ylabel("Observed PIT quantile")
ax.set_title("Reliability (PIT / Q-Q)"); ax.legend()
ax.set_aspect("equal"); plt.tight_layout()

## 4 · Deterministic skill

Standard scores for the ensemble **median** against the observations, alongside HYPE's own best deterministic calibration as a reference.

In [ ]:
def _clean(sim, obs):
    sim = np.asarray(sim, float); obs = np.asarray(obs, float)
    m = np.isfinite(sim) & np.isfinite(obs)
    return sim[m], obs[m]

def nse(sim, obs):
    sim, obs = _clean(sim, obs)
    return 1 - np.sum((sim - obs) ** 2) / np.sum((obs - obs.mean()) ** 2)

def kge(sim, obs):
    sim, obs = _clean(sim, obs)
    r = np.corrcoef(sim, obs)[0, 1]
    beta = sim.mean() / obs.mean()
    gamma = (sim.std() / sim.mean()) / (obs.std() / obs.mean())
    return 1 - np.sqrt((r - 1) ** 2 + (beta - 1) ** 2 + (gamma - 1) ** 2)

def pbias(sim, obs):
    sim, obs = _clean(sim, obs)
    return 100 * (sim.sum() - obs.sum()) / obs.sum()

skill = pd.DataFrame({
    "NSE": [nse(median, obs), nse(best_hype, obs)],
    "KGE": [kge(median, obs), kge(best_hype, obs)],
    "PBIAS [%]": [pbias(median, obs), pbias(best_hype, obs)],
}, index=["GPU_HYPE median", "Best HYPE"]).round(3)
skill

## 5 · Calibrated parameter spread

Because calibration returns a population, each parameter has a *distribution*, not a single value. The trade-off front below (reliability vs error) is the set the forecast is built from.

In [ ]:
fit = model.fitted
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(fit[:, 0], fit[:, 1], s=15, edgecolor="k", alpha=0.6)
ax.set_xlabel("Non-exceedance (reliability)"); ax.set_ylabel("error metric")
ax.set_title("Calibrated population (%d members)" % fit.shape[0])
plt.tight_layout()

## Going further

- The research diagnostics in [`functions2.py`](../functions2.py) (`plot_prediction`, `calculate_metrics`) build richer multi-panel hydrographs and a fuller metric table (CRPS, reliability, resolution). They reuse the companion `forecast_performance` package — install it with `pip install -e ".[metrics]"`.
- Re-run [notebook 02](02_training_and_predicting.ipynb) over a different window, or calibrate your own model in [notebook 01](01_calibration.ipynb) and point this notebook at the resulting pickle.